# Colab'da Faz 2B + 2C Tam Eğitimi (GPU)

Bu defter iki ağır modeli **GPU ile** eğitir — CPU'da anlamsız olan kısım:

| Faz | Model | Girdi | Tahmini süre (T4) |
|---|---|---|---|
| **2B** | Ürkmez 2D taşkın | `h,u,v = f(x,y,t)` | ~30–60 dk |
| **2C** | Parametrik 2D | `h,u,v = f(x,y,t,B,t_f,h0,n)` | ~1–2 saat |

## Kullanım
1. **GPU'yu aç:** `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. Hücreleri sırayla çalıştır (Shift+Enter).
3. Kod GitHub'dan otomatik çekilir — dosya yüklemene gerek yok.

> **Öneri:** Faz 2B'yi çalıştır → sonuçları indir → sonra 2C'ye geç.
> Colab oturumu koparsa dosyalar silinir; her fazdan sonra indirme hücresini çalıştır.

## 1) GPU kontrolü

In [ ]:
import torch
if torch.cuda.is_available():
    print('✓ GPU aktif:', torch.cuda.get_device_name(0))
else:
    print('⚠ GPU YOK! Runtime > Change runtime type > T4 GPU seç ve tekrar çalıştır.')

## 2) Bağımlılıklar

In [ ]:
!pip -q install deepxde pyyaml

## 3) Projeyi GitHub'dan çek
Zip yükleme yok — kod doğrudan repodan geliyor.
Bu hücre tekrar çalıştırılırsa son sürümü çeker (`git pull`).

In [ ]:
import os, subprocess

REPO = 'https://github.com/BurakkYuce/PINN-BAP.git'
if os.path.exists('/content/PINN-BAP'):
    subprocess.run(['git','-C','/content/PINN-BAP','pull','-q'])   # güncelle
else:
    subprocess.run(['git','clone','-q',REPO,'/content/PINN-BAP'])  # ilk çekiş
os.chdir('/content/PINN-BAP')
print('✓ Proje kökü:', os.getcwd())
print('  içerik:', sorted(os.listdir('.')))

## 4) Kurulum kontrolü (saniyeler — eğitim YOK)
Modeller doğru kuruluyor mu, residual'lar hesaplanıyor mu? Saatlerce eğitmeden önce bunu gör.

In [ ]:
!DDE_BACKEND=pytorch python -c "
import sys; sys.path.insert(0,'src')
TINY=dict(adam_iters=2,lbfgs=False,num_domain=200,num_boundary=200,num_initial=100)
from pinn import pinn_parametric_2d as P2
m=P2.build_model(P2.load_config(),TINY); m.compile('adam',lr=1e-3); m.train(iterations=2,display_every=1000)
print('✓ Faz 2C kuruldu')
from pinn import pinn_urkmez_2d as U2
mu,_=U2.build_model(U2.load_config(),TINY); mu.compile('adam',lr=1e-3); mu.train(iterations=2,display_every=1000)
print('✓ Faz 2B kuruldu')
"

---
# FAZ 2B — Ürkmez Barajı 2D taşkın

Gerçek vaka: breach hidrografı (Froehlich: B=63.7 m, t_f=2052 s, Q_pik=6417 m³/s)
kritik akış koşuluyla mansap taşkın ovasına besleniyor.

⚠ **Sınırlama:** taban düz+sabit eğim (gerçek DEM yok) ve varsayılan `T=900 s`
yalnızca hidrografın yükselen kolunu kapsıyor.

### 2B.1 — Hızlı smoke (birkaç dk)

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_urkmez_2d.py --smoke

### 2B.2 — TAM eğitim (asıl iş)

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_urkmez_2d.py

### 2B.3 — FLO-2D karşılaştırması
FLO-2D çıktın varsa `data/raw/urkmez_flo2d_maxdepth.asc` olarak yükle (ESRI ASCII grid)
veya `FLO2D_PATH` ile yol ver. Yoksa betik beklenen formatı yazdırıp çıkar.

In [ ]:
!python validation/compare_urkmez_flo2d.py
import os
from IPython.display import Image, display
p='results/2d/urkmez_pinn_vs_flo2d.png'
if os.path.exists(p): display(Image(p))

### 2B.4 — Sonuçları indir

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/faz2b-urkmez-results','zip','results/2d')
files.download('/content/faz2b-urkmez-results.zip')

---
# FAZ 2C — Parametrik 2D (7 girdi)

Projenin asıl yeniliği: ağ bir senaryo **ailesini** öğrenir.
Rezervuar alan içindedir; baraj gövdesinde **zamanla büyüyen** açıklık
`B(t) = B·min(t/t_f, 1)` bir residual terimi olarak gömülüdür — breach
hidrografı dışarıdan verilmez, akış fizikten doğar.

### 2C.1 — Hızlı smoke

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_parametric_2d.py --smoke

### 2C.2 — TAM eğitim (uzun — 1-2 saat)
7 boyutlu girdi uzayı; sabırlı ol. Kopma riskine karşı sekmeyi açık tut.

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_parametric_2d.py

### 2C.3 — Fizik testleri
Parametrik modelde tek bir analitik referans yoktur; bunun yerine çözümün
sağlaması gereken değişmezler sınanır:
- **TEST 1 — Kütle korunumu:** ∫h dA sabit kalmalı
- **TEST 2 — Parametre duyarlılığı:** B↑ veya t_f↓ ⇒ mansaba daha çok su
- **TEST 3 — Senaryo haritaları**

In [ ]:
!DDE_BACKEND=pytorch python validation/compare_parametric_2d.py
from IPython.display import Image, display
display(Image('results/2d/parametric_2d_scenarios.png'))
display(Image('results/2d/parametric_2d_sensitivity.png'))

### 2C.4 — Sonuçları indir

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/faz2c-parametrik-results','zip','results/2d')
files.download('/content/faz2c-parametrik-results.zip')

---
## Eğitim yetmezse ne ayarlanır?
`config/param_ranges_2d.yaml` içinde:
- `urkmez_2d.domain.T` → pik + resesyon için ~2500 s (çok daha uzun eğitim)
- `*.training.full.adam_iters` → iterasyon sayısı
- `*.training.layers` → ağ kapasitesi
- `parametric_2d.param_ranges` → parametre aralığını daraltmak öğrenmeyi kolaylaştırır
- `nu` → şok yayması/keskinlik dengesi (düşürmek keskinleştirir, kararlılığı zorlar)